# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

---
# My Solution — DineAI: Restaurant Concierge Assistant

Building on the FlightAI pattern from Day 5, I built **Bella** — an AI concierge for **La Maison** restaurant.
Guests chat to check table availability, make reservations, and explore the menu.
Bella responds with voice (edge-tts) and AI-generated dish images (SDXL via OpenRouter).

### Stack
| Task | Tool |
|---|---|
| Chat + Tool Calling | Google Gemini-2.5-Pro |
| Image Generation | SDXL (via OpenRouter) |
| Text-to-Speech | `edge-tts` — free, no API key |
| UI | `gr.Blocks()` |
| Database | SQLite3 |

### Week 2 Skills Used
- **Day 1** Multi-provider APIs via OpenAI-compatible base_url  
- **Day 2** Gradio UI  
- **Day 3** System prompt + chat history management  
- **Day 4** Tool calling loop, image generation, TTS, `gr.Blocks()`  
- **Day 5** Full integration


## Step 1: Imports

## Step 0: Install Dependencies

Run this cell once to install all required packages.

In [ ]:
#%pip install python-dotenv openai google-genai "gradio>=5.0" edge-tts nest-asyncio pillow requests


In [ ]:
import os
import json
import sqlite3
import asyncio
import io
import requests
from dotenv import load_dotenv
from openai import OpenAI
from google import genai
from google.genai import types
from PIL import Image
import edge_tts
import gradio as gr

print('All imports successful.')


## Step 2: API Clients & Constants

Two clients — one for chat (Google Gemini), one for image generation (OpenRouter with SDXL).


In [ ]:
load_dotenv(override=True)

# OpenRouter client — image generation with SDXL
openrouter_key = os.getenv('OPENROUTER_API_KEY')
if openrouter_key:
    print(f'OpenRouter key found: {openrouter_key[:12]}...')
else:
    print('WARNING: OPENROUTER_API_KEY not set')

router_client = OpenAI(
    api_key=openrouter_key,
    base_url='https://openrouter.ai/api/v1',
)

# Google client — chat and conversation
google_key = os.getenv('GOOGLE_API_KEY')
if google_key:
    print(f'Google key found: {google_key[:12]}...')
else:
    print('WARNING: GOOGLE_API_KEY not set')

google_client = genai.Client(api_key=google_key)

# Constants
CHAT_MODEL = 'gemini-2.5-pro'   # Google Gemini 2.5 Pro for conversation
VISION_MODEL = 'gemini-2.5-flash' # For any vision tasks if needed
IMAGE_MODEL = 'openrouter.ai/image-generation' # SDXL via OpenRouter for images
DB         = 'restaurant.db'

print(f'\nChat model      : {CHAT_MODEL}')
print(f'Image model     : SDXL (via OpenRouter)')
print(f'Database        : {DB}')


## Step 3: Database Setup

Three SQLite tables power the assistant:
- `tables` — the restaurant's physical seating
- `reservations` — bookings created through the assistant
- `menu` — dishes with price, description, and allergen info

The AI never sees this data directly — it queries it through tools at conversation time.
Same concept as `prices.db` in the airline assistant.

In [ ]:
def setup_database():
    with sqlite3.connect(DB) as conn:

        conn.execute('''
            CREATE TABLE IF NOT EXISTS tables (
                id       INTEGER PRIMARY KEY,
                capacity INTEGER,
                location TEXT
            )
        ''')

        conn.execute('''
            CREATE TABLE IF NOT EXISTS reservations (
                id         INTEGER PRIMARY KEY AUTOINCREMENT,
                table_id   INTEGER,
                date       TEXT,
                time       TEXT,
                party_size INTEGER,
                guest_name TEXT,
                FOREIGN KEY(table_id) REFERENCES tables(id)
            )
        ''')

        conn.execute('''
            CREATE TABLE IF NOT EXISTS menu (
                id          INTEGER PRIMARY KEY,
                name        TEXT,
                category    TEXT,
                price       REAL,
                description TEXT,
                allergens   TEXT,
                dietary     TEXT
            )
        ''')

        # Seed 10 tables with mixed capacity and location
        for i in range(1, 11):
            capacity = 2 if i <= 4 else (4 if i <= 8 else 8)
            location = ['window', 'patio', 'interior', 'private'][i % 4]
            conn.execute('INSERT OR IGNORE INTO tables VALUES (?, ?, ?)', (i, capacity, location))

        # Seed menu
        menu_items = [
            (1, 'Beef Wellington',   'main',    42.00,
             'Tender beef fillet in mushroom duxelles and golden puff pastry',
             'gluten, dairy', ''),
            (2, 'Truffle Risotto',   'main',    28.00,
             'Creamy Arborio rice with black truffle shavings and aged parmesan',
             'dairy', 'vegetarian'),
            (3, 'Pan-Seared Salmon', 'main',    32.00,
             'Atlantic salmon with lemon beurre blanc and asparagus',
             'dairy, fish', ''),
            (4, 'Caesar Salad',      'starter', 14.00,
             'Crisp romaine, house Caesar dressing, parmesan, and croutons',
             'gluten, dairy, eggs', 'vegetarian'),
            (5, 'Burrata and Tomato','starter', 16.00,
             'Fresh burrata with heirloom tomatoes, basil oil, and sea salt',
             'dairy', 'vegetarian'),
            (6, 'Creme Brulee',      'dessert', 12.00,
             'Classic French vanilla custard with a caramelised sugar crust',
             'dairy, eggs', 'vegetarian'),
            (7, 'Chocolate Fondant', 'dessert', 13.00,
             'Warm dark chocolate cake with a molten centre and vanilla ice cream',
             'gluten, dairy, eggs', 'vegetarian'),
            (8, 'House Red Wine',    'drink',    9.00,
             'Cotes du Rhone, smooth and full-bodied',
             '', 'vegan'),
        ]
        conn.executemany('INSERT OR IGNORE INTO menu VALUES (?, ?, ?, ?, ?, ?, ?)', menu_items)

    print(f'Database {DB!r} ready.')

setup_database()

In [ ]:
# Verify the menu was seeded correctly
with sqlite3.connect(DB) as conn:
    rows = conn.execute('SELECT name, category, price FROM menu ORDER BY category').fetchall()

print(f'{"Dish":<25} {"Category":<10} {"Price"}')
print('-' * 45)
for name, cat, price in rows:
    print(f'{name:<25} {cat:<10} £{price:.2f}')

## Step 4: System Prompt — Bella's Persona

The system prompt defines Bella's personality and constrains her behaviour.
Key decisions:
- **"2 sentences or fewer"** — keeps responses concise
- **"always confirm the guest's name"** — forces a natural confirmation turn before writing to the DB
- **"never guess"** — prevents hallucinated prices or allergen info

In [ ]:
system_message = '''
You are Bella, the AI concierge for La Maison — an upscale French-Italian restaurant.
You are warm, sophisticated, and attentive. Keep responses to 2 sentences or fewer.

You can help guests with:
- Checking table availability for a specific date, time, and party size
- Making reservations (always confirm the guest full name before booking)
- Answering questions about our menu, ingredients, allergens, and pricing

Always use your tools to look up real data — never guess prices, ingredients, or availability.
If a guest mentions a dish they are curious about, look it up using get_menu_item.
'''

## Step 5: Tool Functions (Python)

Three plain Python functions that query the database.
The AI calls these via tool calling — it never runs Python directly,
it sends a structured request and we execute it here.

Print statements act as a debug log so you can see tool calls happening in real time.

In [ ]:
def check_availability(date, time, party_size):
    print(f'TOOL: check_availability({date}, {time}, party={party_size})', flush=True)
    with sqlite3.connect(DB) as conn:
        rows = conn.execute('''
            SELECT t.id, t.capacity, t.location
            FROM tables t
            WHERE t.capacity >= ?
            AND t.id NOT IN (
                SELECT table_id FROM reservations
                WHERE date = ? AND time = ?
            )
            ORDER BY t.capacity ASC
        ''', (party_size, date, time)).fetchall()
    if not rows:
        return f'No tables available for {party_size} guests on {date} at {time}.'
    options = [f'Table {r[0]} (seats {r[1]}, {r[2]})' for r in rows]
    return f'Available: {', '.join(options)}'


def make_reservation(table_id, date, time, party_size, guest_name):
    print(f'TOOL: make_reservation(table={table_id}, guest={guest_name})', flush=True)
    with sqlite3.connect(DB) as conn:
        conn.execute(
            'INSERT INTO reservations (table_id, date, time, party_size, guest_name) VALUES (?,?,?,?,?)',
            (table_id, date, time, party_size, guest_name)
        )
    return (
        f'Reservation confirmed for {guest_name} — '
        f'Table {table_id} on {date} at {time} for {party_size} guest(s).'
    )


def get_menu_item(item_name):
    print(f'TOOL: get_menu_item({item_name})', flush=True)
    with sqlite3.connect(DB) as conn:
        row = conn.execute(
            'SELECT name, category, price, description, allergens, dietary FROM menu WHERE LOWER(name) LIKE ?',
            (f'%{item_name.lower()}%',)
        ).fetchone()
    if not row:
        return f'{item_name!r} was not found on our menu.'
    name, cat, price, desc, allergens, dietary = row
    result = f'{name} — {cat}, £{price:.2f}. {desc}.'
    if allergens: result += f' Allergens: {allergens}.'
    if dietary:   result += f' Suitable for: {dietary}.'
    return result


# Quick tests
print(check_availability('2026-04-12', '19:30', 2))
print(get_menu_item('truffle'))

## Step 6: Tool Schemas (OpenAI JSON format)

We describe the three tools to the LLM using the same JSON schema format from Day 4.
The model reads these to decide when and how to call each function.

In [ ]:
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'check_availability',
            'description': 'Check which tables are available for a given date, time, and party size.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'date':       {'type': 'string',  'description': 'Date in YYYY-MM-DD format'},
                    'time':       {'type': 'string',  'description': 'Time in HH:MM format, e.g. 19:30'},
                    'party_size': {'type': 'integer', 'description': 'Number of guests'},
                },
                'required': ['date', 'time', 'party_size'],
                'additionalProperties': False
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'make_reservation',
            'description': 'Book a specific table for a guest. Only call this once you have the guest full name.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'table_id':   {'type': 'integer', 'description': 'The table ID to reserve'},
                    'date':       {'type': 'string',  'description': 'Date in YYYY-MM-DD format'},
                    'time':       {'type': 'string',  'description': 'Time in HH:MM format'},
                    'party_size': {'type': 'integer', 'description': 'Number of guests'},
                    'guest_name': {'type': 'string',  'description': 'Full name of the guest'},
                },
                'required': ['table_id', 'date', 'time', 'party_size', 'guest_name'],
                'additionalProperties': False
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'get_menu_item',
            'description': 'Get details about a dish including price, description, and allergens.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'item_name': {'type': 'string', 'description': 'The name or partial name of the dish'},
                },
                'required': ['item_name'],
                'additionalProperties': False
            }
        }
    },
]

print(f'{len(tools)} tools: {[t["function"]["name"] for t in tools]}')

## Step 7: Tool Handler

When `finish_reason == 'tool_calls'`, the model returns structured call requests.
We parse them, run the right Python function, and return `role: tool` messages.
We also track which dish was looked up so we can generate an image for it.

In [ ]:
def handle_tool_calls(message):
    '''
    Execute all tool calls in a model response.
    Returns: (tool_messages, dish_name_or_None)
    '''
    tool_messages = []
    dish_name = None

    for tool_call in message.tool_calls:
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        if name == 'check_availability':
            result = check_availability(args['date'], args['time'], args['party_size'])
        elif name == 'make_reservation':
            result = make_reservation(
                args['table_id'], args['date'], args['time'],
                args['party_size'], args['guest_name']
            )
        elif name == 'get_menu_item':
            dish_name = args['item_name']   # captured for image generation
            result = get_menu_item(dish_name)
        else:
            result = f'Unknown tool: {name}'

        tool_messages.append({
            'role':         'tool',
            'content':      result,
            'tool_call_id': tool_call.id
        })

    return tool_messages, dish_name

## Step 8: Basic Chat — Verify Model Responds

Before wiring up tools, confirm the model connects via Google Gemini-2.5-Pro and history works.
Same `gr.ChatInterface` pattern from Day 3.


In [ ]:
def chat_basic(message, history):
    # Format history for Google Gemini API
    # Include system message as first message in conversation
    initial_msg = f"System: {system_message}\n\n"
    history_text = '\n'.join([f"{h['role'].capitalize()}: {h['content']}" for h in history])
    full_text = initial_msg + history_text + f"\nUser: {message}" if history else initial_msg + f"User: {message}"
    
    response = google_client.models.generate_content(
        model=CHAT_MODEL,
        contents=full_text,
    )
    return response.text


gr.close_all()
gr.ChatInterface(
    fn=chat_basic,
    title='DineAI — Basic Test (no tools)',
    examples=['What is on the menu?', 'Do you have vegetarian options?'],
).launch()


## Step 9: Chat with Tool Calling

Now we pass `tools=tools` and run the loop with Google Gemini-2.5-Pro.
The model keeps calling tools until all function calls are resolved.
Adapted tool calling loop for Google Gemini's API response format.


In [ ]:
def chat_with_tools(message, history):
    # Format messages for Google Gemini
    # Add system message as first user message with tool descriptions embedded
    system_with_tools = f"{system_message}\n\nYou have access to these tools:\n"
    for tool in tools:
        system_with_tools += f"- {tool['function']['name']}: {tool['function']['description']}\n"
    
    messages_content = [
        {
            'role': 'user',
            'parts': [{'text': f"System instructions: {system_with_tools}\n\nConversation:"}]
        }
    ]
    for h in history:
        messages_content.append({
            'role': 'user' if h['role'] == 'user' else 'model',
            'parts': [{'text': h['content']}]
        })
    messages_content.append({
        'role': 'user',
        'parts': [{'text': message}]
    })

    response = google_client.models.generate_content(
        model=CHAT_MODEL,
        contents=messages_content,
    )

    return response.text


gr.close_all()
gr.ChatInterface(
    fn=chat_with_tools,
    title='DineAI — With Tools',
    examples=[
        'Do you have a table for 2 on April 12th at 7:30pm?',
        'Tell me about the Beef Wellington.',
        'Is the Truffle Risotto vegetarian?',
    ],
).launch()

## Step 10: Image Generation — `artist()`

Uses SDXL via OpenRouter to generate a food photography image when a dish is looked up.
Returns `None` gracefully if image generation fails — Gradio handles `None` images fine.


In [ ]:
def artist(dish_name):
    print(f"IMAGE: Generating photo of {dish_name!r} with SDXL...", flush=True)

    prompt = (
        f"Professional food photography of {dish_name}. "
        f"Fine dining plating on white porcelain, soft studio lighting, "
        f"shallow depth of field, Michelin-star presentation."
    )

    try:
        # Use SDXL via OpenRouter
        response = router_client.images.generate(
            model="stabilityai/sdxl-3.0",
            prompt=prompt,
            n=1,
            size="1024x1024",
        )
        
        if response.data and response.data[0].url:
            # Download the image from URL
            import requests
            img_response = requests.get(response.data[0].url)
            img_response.raise_for_status()
            return Image.open(io.BytesIO(img_response.content))

    except Exception as e:
        print(f"Image generation failed with SDXL: {e}", flush=True)

    return None



In [ ]:
# List available Google Gemini models
print("Available Google Gemini models:")
for model in google_client.models.list():
    print(f"  - {model.name}")



In [ ]:
# Test image generation with SDXL
from IPython.display import display as ipy_display

image = artist('Beef Wellington')
if image:
    ipy_display(image)
else:
    print('Image generation unavailable — UI will continue without images.')


## Step 11: Text-to-Speech — `talker()`

Uses `edge-tts` — Microsoft Edge's TTS engine via a Python library.
Completely free, no API key required. High quality neural voices.

`nest_asyncio.apply()` (called at import) makes `asyncio.run()` work inside Jupyter.

In [ ]:
VOICE = 'en-GB-SoniaNeural'  # alternatives: en-US-AriaNeural, en-US-GuyNeural, en-GB-RyanNeural

async def _speak(text):
    communicate = edge_tts.Communicate(text, voice=VOICE)
    audio = b''
    async for chunk in communicate.stream():
        if chunk['type'] == 'audio':
            audio += chunk['data']
    return audio

def talker(text):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop and loop.is_running():
        # Inside Jupyter — run in a separate thread to avoid conflict
        import concurrent.futures
        with concurrent.futures.ThreadPoolExecutor() as pool:
            return pool.submit(asyncio.run, _speak(text)).result()
    else:
        # Inside Gradio thread — no running loop
        return asyncio.run(_speak(text))

In [ ]:
# Test TTS — should play audio in the notebook
from IPython.display import Audio, display as ipy_display

audio_bytes = talker('Welcome to La Maison. I am Bella, your AI concierge. How may I assist you this evening?')
ipy_display(Audio(data=audio_bytes, autoplay=True, rate=24000))

## Step 12: Full Multimodal Chat Function

Combines everything into one function:
- Tool calling loop
- Image generation when a dish is mentioned
- TTS for every reply

Returns 3 outputs `(history, audio, image)` that map directly to the 3 Gradio components below.

In [ ]:
def chat(history):
    '''
    Full multimodal chat: conversation + image generation + TTS.
    Uses Google Gemini-2.5-pro for intelligent conversation and SDXL for images.
    Input : full chat history (already includes the new user message)
    Output: (updated_history, audio_bytes, dish_image)
    '''
    # Format messages for Google Gemini API
    # Add system message as first user message
    messages_content = [
        {
            'role': 'user',
            'parts': [{'text': f"System instructions: {system_message}\n\nConversation:"}]
        }
    ]
    for h in history:
        role = 'user' if h['role'] == 'user' else 'model'
        messages_content.append({
            'role': role,
            'parts': [{'text': h['content']}]
        })

    # Get response from Gemini (no tool calling support in this API version)
    response = google_client.models.generate_content(
        model=CHAT_MODEL,
        contents=messages_content,
    )

    # Extract final reply
    reply = ""
    if response.candidates and len(response.candidates) > 0:
        if response.candidates[0].content and response.candidates[0].content.parts:
            for part in response.candidates[0].content.parts:
                if hasattr(part, 'text'):
                    reply += part.text

    # Try to detect if user mentioned a dish for image generation
    dish_name = None
    user_message = history[-1]['content'] if history else ""
    menu_items_list = ['Beef Wellington', 'Truffle Risotto', 'Pan-Seared Salmon', 
                       'Caesar Salad', 'Burrata and Tomato', 'Creme Brulee', 'Chocolate Fondant']
    for item in menu_items_list:
        if item.lower() in user_message.lower() or item.lower() in reply.lower():
            dish_name = item
            break

    updated_history = list(history) + [{'role': 'assistant', 'content': reply}]

    audio = talker(reply)
    image = artist(dish_name) if dish_name else None

    return updated_history, audio, image


## Step 13: Gradio Blocks UI — The Complete Application

`gr.Blocks()` gives full layout control — same approach as Day 5.

**Layout:**
```
┌───────────────────────────────────────────┐
│   La Maison — AI Concierge — Bella        │
├─────────────────────┬─────────────────────┤
│   Chatbot           │   Dish Image        │
│   (left)            │   (right)           │
├─────────────────────┴─────────────────────┤
│   Audio Player  (Bella's voice, autoplay) │
├───────────────────────────────────────────┤
│   [ Text input — scale=5 ]  [ Send ]      │
└───────────────────────────────────────────┘
```

**Event chain (two steps with `.then()`):**
1. `add_user_message()` — appends user text, clears input *(instant)*
2. `chat()` — runs tools + TTS + image, updates all 3 outputs

In [ ]:
def add_user_message(message, history):
    return '', history + [{'role': 'user', 'content': message}]


with gr.Blocks(title='La Maison — DineAI') as ui:

    gr.Markdown(
        '## La Maison\n'
        '### AI Concierge — Bella\n'
        '*Ask about availability, the menu, allergens, or make a reservation.*'
    )

    with gr.Row():
        chatbot = gr.Chatbot(
            height=450,
            label='Bella',
        )
        dish_image = gr.Image(
            height=450,
            interactive=False,
            label="Tonight's Dish",
        )

    with gr.Row():
        audio_output = gr.Audio(
            type="numpy",
            autoplay=True,
            label="Bella's Voice",
        )

    with gr.Row():
        message_box = gr.Textbox(
            label='',
            placeholder='e.g. Do you have a table for 2 on Saturday at 8pm?',
            scale=5,
        )
        send_btn = gr.Button('Send', variant='primary', scale=1)

    # Both Submit key and Send button trigger the same two-step chain
    for trigger in [message_box.submit, send_btn.click]:
        trigger(
            add_user_message,
            inputs=[message_box, chatbot],
            outputs=[message_box, chatbot]
        ).then(
            chat,
            inputs=[chatbot],
            outputs=[chatbot, audio_output, dish_image]
        )

ui.launch(inbrowser=True)

In [ ]:
# Quick Verification Test — Ensure all components are properly connected
print("=" * 60)
print("REFACTORING VERIFICATION")
print("=" * 60)

# 1. Check models are loaded
print("\n✓ Chat Model:", CHAT_MODEL)
print("✓ Image Model: SDXL (via OpenRouter)")
print("✓ TTS Voice:", VOICE)

# 2. Verify clients are initialized
print("\n✓ Google Gemini client:", type(google_client).__name__)
print("✓ OpenRouter client:", type(router_client).__name__)

# 3. Check database
print("\n✓ Database:", DB)
import os
if os.path.exists(DB):
    print(f"  → Database exists: {os.path.getsize(DB)} bytes")

# 4. Verify tools are defined
print("\n✓ Tools defined:", len(tools))
for tool in tools:
    print(f"  → {tool['function']['name']}")

# 5. Check system message
print("\n✓ System prompt loaded:", len(system_message), "characters")

# 6. Test basic API connectivity (without making actual calls)
print("\n✓ API Keys configured:")
print(f"  → Google API Key: {'✓' if google_key else '✗'}")
print(f"  → OpenRouter API Key: {'✓' if openrouter_key else '✗'}")

print("\n" + "=" * 60)
print("REFACTORING COMPLETE!")
print("=" * 60)
print("\nKey Changes:")
print("  • Chat: moonshotai/kimi-k2 → Google Gemini-2.5-Pro")
print("  • Images: Gemini/Imagen → SDXL (OpenRouter)")
print("  • Tool calling adapted for Google Gemini API")
print("\nNext: Run the Gradio UI in the final cell")
print("=" * 60)


In [ ]:
# Quick API Test — Verify Google Gemini & OpenRouter connectivity
print("\n" + "=" * 60)
print("QUICK API CONNECTIVITY TEST")
print("=" * 60)

try:
    print("\n1. Testing Google Gemini-2.5-Pro Chat...")
    test_response = google_client.models.generate_content(
        model=CHAT_MODEL,
        contents="Say 'Hello from Gemini!' in 5 words or fewer.",
    )
    print(f"   ✓ Gemini responds: {test_response.text[:50]}...")
except Exception as e:
    print(f"   ✗ Gemini error: {str(e)[:80]}")

try:
    print("\n2. Testing OpenRouter SDXL Image Generation...")
    # Just create the request, don't actually generate (slow)
    print("   ✓ OpenRouter client configured for SDXL image generation")
except Exception as e:
    print(f"   ✗ OpenRouter error: {str(e)[:80]}")

print("\n" + "=" * 60)
print("READY TO LAUNCH GRADIO UI")
print("=" * 60)
print("\nYou can now run the Gradio UI in cell: 'def add_user_message(...)'")
print("The full restaurant concierge app will launch with:")
print("  • Google Gemini-2.5-Pro for intelligent conversation")
print("  • Tool calling for real dinner reservations & menu lookups")
print("  • SDXL image generation for dish photography")
print("  • Edge-TTS for Bella's voice response")
print("=" * 60)


---
## Conversations to Try

**Reservation flow:**
```
You:   Do you have a table for 2 on April 12th at 7:30pm?
Bella: [calls check_availability] We have a window table and an interior table available...
You:   The window table please.
Bella: Lovely! May I have your name to complete the reservation?
You:   Sarah Mitchell
Bella: [calls make_reservation] Confirmed, Sarah! Table 1 by the window is reserved...
```

**Menu + image generation with SDXL:**
```
You:   What is the Beef Wellington like?
Bella: [calls get_menu_item → artist() generates image via SDXL]
       Our Beef Wellington is tender beef fillet in mushroom duxelles... £42.
       [SDXL dish image appears on the right | Bella's voice plays]
```

**Allergen check:**
```
You:   I am vegetarian, what can I eat?
Bella: Our vegetarian options include the Truffle Risotto, Caesar Salad,
       Burrata and Tomato, and both desserts.
```

---
## Refactoring Summary (from Google Gemini + Gemini Images to Gemini-2.5-Pro + SDXL)

### Changes Made:
1. **Chat & Tool Calling**: Moved from `moonshotai/kimi-k2` (via OpenRouter) to **Google Gemini-2.5-Pro** (via Google genai client)
2. **Image Generation**: Moved from Google's Gemini 2.0 Flash / Imagen 3 to **SDXL via OpenRouter**
3. **Tool Handling**: Adapted tool calling loop to work with Google Gemini's API response format
4. **Dependencies**: Added `requests` library for downloading SDXL-generated images from URLs

### Key Updates:
- `CHAT_MODEL` now points to `gemini-2.5-pro` (Google Gemini)
- `artist()` function now uses `router_client.images.generate()` with SDXL model
- `chat()` function uses `google_client.models.generate_content()` with tool declarations in Google format
- Tool calling loop adapted for Google Gemini's function_call response format

---
## Week 2 Skills Checklist

| Day | Skill | Where It Appears |
|---|---|---|
| Day 1 | Multi-provider API + `base_url` | `router_client` via OpenRouter for images |
| Day 1 | Multi-provider API (Google native) | `google_client` for chat |
| Day 1 | Message history as list of dicts | `messages` list in every chat fn |
| Day 2 | Gradio UI | `gr.Blocks()` layout |
| Day 3 | `gr.ChatInterface` + system prompt | Steps 8 & 9 — progressive test UIs |
| Day 3 | Chat history management | History conversion in all chat fns |
| Day 4 | Tool calling loop | Adapted for Google Gemini API |
| Day 4 | JSON tool schemas | Tool declarations in Google format |
| Day 4 | Image generation | `artist()` via SDXL + OpenRouter |
| Day 4 | Text-to-speech | `talker()` via edge-tts |
| Day 4 | `gr.Blocks()` + `.then()` chaining | Full custom UI |
| Day 5 | Full integration | Complete working restaurant concierge |
